# Semana 4 — Ordenamiento, greedy y divide y vencerás aplicados a Penonomé

Aplica las unidades 27 (burbuja, selección e inserción), 28 (fusión), 29 (ordenamiento rápido), 30 (enfoque greedy) y 31 (divide y vencerás) a los datos oficiales del proyecto.

[Proyecto](../../README.md) | [Anterior](s03-pilas-colas-busqueda-hash.ipynb) | [Siguiente](s05-dinamica-backtracking-modulos-pandas.ipynb)

In [1]:
def verificar(descripcion, obtenido, esperado):
    """Compara el resultado de un ejercicio con el esperado e informa sin detener el notebook."""
    if obtenido == esperado:
        print(f"Correcto: {descripcion}")
    else:
        print(f"Incorrecto: {descripcion}. Se esperaba {esperado!r} y se obtuvo {obtenido!r}")

## Datos

- `02_censo2023_viviendas_caracteristicas.csv` (filas de nivel corregimiento): viviendas y viviendas sin internet de los 16 corregimientos. Se ordenan por el porcentaje de viviendas sin internet.
- `01_censo2023_poblacion_corregimientos.csv`: población de 2010 y 2023 y superficie, para ordenar por crecimiento.
- `99_base_larga_todo.csv`: las 460 filas de la base larga (Censo 2023 e índice de pobreza multidimensional), para medir los algoritmos con un volumen mayor.
- `datos/analisis/replicas_acumuladas.csv`: réplicas acumuladas del sismo del 9 de octubre de 2026 en siete cortes, según la prensa con datos del IGC-UP.

Los algoritmos viven en `penonome.ordenamiento`. Cada función recibe la lista y una `clave`, no modifica la original y devuelve `(lista_ordenada, comparaciones)`.

In [2]:
import inspect
from datetime import date
from math import log2

from penonome import ANALISIS
from penonome.carga import PADRES_LEY_296, cargar_base_larga, cargar_poblacion, cargar_viviendas, leer_csv
from penonome.busqueda import find_two
from penonome.ordenamiento import ALGORITMOS, burbuja, fusion, insercion, medir, rapido, seleccion

viviendas = cargar_viviendas("corregimiento")
distrito = cargar_viviendas("distrito")[0]
poblacion = cargar_poblacion()
base = cargar_base_larga()
replicas = leer_csv(ANALISIS / "replicas_acumuladas.csv", numericas=("replicas_acumuladas",))

corregimientos = []
for fila in viviendas:
    corregimientos.append({
        "id": fila["id_corregimiento"],
        "corregimiento": fila["area"],
        "viviendas": fila["viviendas"],
        "sin_internet": fila["sin_internet"],
        "pct_sin_internet": 100 * fila["sin_internet"] / fila["viviendas"],
    })

print("Corregimientos:", len(corregimientos), "| filas de la base larga:", len(base), "| cortes de réplicas:", len(replicas))
verificar("16 corregimientos", len(corregimientos), 16)
verificar("460 filas en la base larga", len(base), 460)
verificar("7 cortes de réplicas", len(replicas), 7)
verificar("las viviendas sin internet suman el total del distrito",
          sum(c["sin_internet"] for c in corregimientos), distrito["sin_internet"])

Corregimientos: 16 | filas de la base larga: 460 | cortes de réplicas: 7
Correcto: 16 corregimientos
Correcto: 460 filas en la base larga
Correcto: 7 cortes de réplicas
Correcto: las viviendas sin internet suman el total del distrito


## 1. Burbuja, selección e inserción

**Referencia:** [Unidad 27 — Burbuja, selección e inserción](../../../../01-python/semana-04/clase-1-algoritmos-de-ordenamiento/u27-burbuja-seleccion-insercion.ipynb). Los tres algoritmos cuadráticos del curso:

- **Burbuja:** compara vecinos e intercambia los invertidos; en cada pasada el mayor llega al final.
- **Selección:** busca el menor de la parte sin ordenar y lo coloca en la posición `i`.
- **Inserción:** toma cada elemento y desplaza a la derecha los mayores hasta encontrar su lugar.

Se ordenan los corregimientos de mayor a menor porcentaje de viviendas sin internet. Para ordenar de mayor a menor con funciones que ordenan de menor a mayor se niega la clave; el nombre desempata.

In [3]:
print(inspect.getsource(insercion))

ranking = lambda c: (-c["pct_sin_internet"], c["corregimiento"])
esperado = sorted(corregimientos, key=ranking)

for nombre, funcion in (("burbuja", burbuja), ("selección", seleccion), ("inserción", insercion)):
    ordenada, comparaciones = funcion(corregimientos, clave=ranking)
    print(f"{nombre:<10} comparaciones = {comparaciones:>4}")
    verificar(f"{nombre} da el mismo orden que sorted()", ordenada, esperado)

ordenada, _ = insercion(corregimientos, clave=ranking)
print()
for posicion, c in enumerate(ordenada, start=1):
    print(f"{posicion:>2}. {c['corregimiento']:<28} {c['pct_sin_internet']:5.1f} % sin internet")
verificar("la lista original no se modificó", corregimientos[0]["corregimiento"], "Penonomé (cabecera)")
verificar("Tulú tiene la mayor proporción sin internet", ordenada[0]["corregimiento"], "Tulú")

def insercion(lista, clave=_identidad):
    """Inserción (`insertionsort2` de la unidad 27): desplaza los mayores e inserta x en su lugar."""
    S = list(lista)
    comparaciones = 0
    for i in range(1, len(S)):
        x = S[i]
        j = i - 1
        while j >= 0:
            comparaciones += 1          # se evalúa S[j] > x
            if not clave(S[j]) > clave(x):
                break
            S[j + 1] = S[j]
            j -= 1
        S[j + 1] = x
    return S, comparaciones

burbuja    comparaciones =  114
Correcto: burbuja da el mismo orden que sorted()
selección  comparaciones =  120
Correcto: selección da el mismo orden que sorted()
inserción  comparaciones =   97
Correcto: inserción da el mismo orden que sorted()

 1. Tulú                          99.4 % sin internet
 2. Río Indio                     99.0 % sin internet
 3. Boca de Tucué                 98.6 % sin internet
 4. Riecito                       98.2 % sin internet
 5. San Miguel                    94.9 % 

**Clave:** con n = 16, selección siempre hace n(n-1)/2 = 120 comparaciones, sin importar el orden de entrada. Burbuja (con la parada temprana) e inserción hacen menos cuando la entrada ya está parcialmente ordenada; en este caso el archivo viene ordenado de corregimientos urbanos a rurales, que es casi el orden inverso del ranking.

**Error común:** creer que los tres algoritmos cuadráticos cuestan lo mismo. Comparten O(n²) en el peor caso, pero inserción hace n - 1 comparaciones con una lista ya ordenada y selección sigue haciendo n(n-1)/2.

## 2. Ordenamiento por fusión (merge sort)

**Referencia:** [Unidad 28 — Ordenamiento por fusión](../../../../01-python/semana-04/clase-1-algoritmos-de-ordenamiento/u28-ordenamiento-por-fusion.ipynb). Divide la lista en dos mitades, ordena cada una de forma recursiva y las mezcla comparando sus primeros elementos. Siempre cuesta O(n log n), a cambio de memoria adicional para las listas intermedias.

Se aplica al crecimiento de la población entre 2010 y 2023. Por la Ley 296 de 2022, los cinco corregimientos divididos y los seis creados a partir de ellos no son comparables por separado: se agrupan como un solo bloque, como indica el LEEME del paquete.

In [4]:
print(inspect.getsource(fusion))

por_id = {c["id_corregimiento"]: c for c in poblacion}
del_bloque = set(PADRES_LEY_296) | {hijo for hijos in PADRES_LEY_296.values() for hijo in hijos}

unidades = [{"unidad": c["corregimiento"], "pob_2010": c["pob_2010"], "pob_2023": c["pob_2023"]}
            for c in poblacion if c["id_corregimiento"] not in del_bloque]
unidades.append({
    "unidad": "Bloque Ley 296 (11 corregimientos)",
    "pob_2010": sum(por_id[i]["pob_2010"] for i in PADRES_LEY_296),
    "pob_2023": sum(por_id[i]["pob_2023"] for i in del_bloque),
})
for u in unidades:
    u["crecimiento_pct"] = 100 * (u["pob_2023"] / u["pob_2010"] - 1)

crecimiento, comparaciones = fusion(unidades, clave=lambda u: -u["crecimiento_pct"])
for u in crecimiento:
    print(f"{u['unidad']:<36} {u['pob_2010']:>7,} -> {u['pob_2023']:>7,}  {u['crecimiento_pct']:+6.1f} %")
print("Comparaciones de la fusión:", comparaciones)

verificar("el bloque reúne 11 corregimientos", len(del_bloque), 11)
verificar("las 6 unidades suman la población 2023 del distrito", sum(u["pob_2023"] for u in unidades), distrito["poblacion"])
verificar("la fusión coincide con sorted()", crecimiento, sorted(unidades, key=lambda u: -u["crecimiento_pct"]))
verificar("El Coco es la unidad que más creció", crecimiento[0]["unidad"], "El Coco")

def fusion(lista, clave=_identidad):
    """Fusión o merge sort (unidad 28): divide en mitades, ordena cada una y las mezcla."""
    contador = [0]

    def mezclar(a, b):
        resultado = []
        i = j = 0
        while i < len(a) and j < len(b):
            contador[0] += 1
            if clave(a[i]) <= clave(b[j]):
                resultado.append(a[i])
                i += 1
            else:
                resultado.append(b[j])
                j += 1
        resultado.extend(a[i:])
        resultado.extend(b[j:])
        return resultado

    def merge_sort(S):
        if len(S) <= 1:
            return list(S)
        mitad = len(S) // 2
        return mezclar(merge_sort(S[:mitad]), merge_sort(S[mitad:]))

    return merge_sort(list(lista)), contador[0]

El Coco                                5,605 ->  11,064   +97.4 %
Coclé                                  4,100 ->   5,811   +41.7 %
Cañaveral                              7,517 ->   9,324   +24.0 %
Penonomé (cabecera)    

## 3. Ordenamiento rápido (quicksort)

**Referencia:** [Unidad 29 — Ordenamiento rápido](../../../../01-python/semana-04/clase-1-algoritmos-de-ordenamiento/u29-ordenamiento-rapido.ipynb). Elige un pivote, deja a su izquierda los menores y a su derecha los mayores (partición) y repite en cada lado. El promedio es O(n log n), pero si el pivote es siempre el menor o el mayor, cada partición solo separa un elemento y el costo sube a O(n²).

`rapido` usa la partición `partition1` corregida de la unidad 29 con el pivote en el centro del rango. Para ver el peor caso se compara con la versión de primer pivote sobre valores que ya vienen ordenados.

In [5]:
print(inspect.getsource(rapido))

def rapido(lista, clave=_identidad):
    """Ordenamiento rápido o quicksort (unidad 29), in situ sobre una copia.

    Usa la partición `partition1` corregida de la unidad 29 (`while left <= right`) con el
    elemento del medio como pivote, para no caer en el peor caso O(n^2) con datos que ya
    vienen casi ordenados.
    """
    S = list(lista)
    contador = [0]

    def partition(low, high):
        medio = (low + high) // 2
        S[low], S[medio] = S[medio], S[low]
        pivot = clave(S[low])
        left, right = low + 1, high
        while left <= right:
            while left <= right:
                contador[0] += 1
                if clave(S[left]) <= pivot:
                    left += 1
                else:
                    break
            while left <= right:
                contador[0] += 1
                if clave(S[right]) >= pivot:
                    right -= 1
                else:
                    break
            if left < right:
                S[le

In [6]:
def quicksort_primer_pivote(lista):
    """Quicksort con el primer elemento como pivote (partition1 corregida); devuelve (S, comparaciones)."""
    S = list(lista)
    cont = [0]

    def partition(low, high):
        pivot = S[low]
        left, right = low + 1, high
        while left <= right:
            while left <= right:
                cont[0] += 1
                if S[left] <= pivot:
                    left += 1
                else:
                    break
            while left <= right:
                cont[0] += 1
                if S[right] >= pivot:
                    right -= 1
                else:
                    break
            if left < right:
                S[left], S[right] = S[right], S[left]
        S[low], S[right] = S[right], S[low]
        return right

    def quicksort(low, high):
        if low < high:
            p = partition(low, high)
            quicksort(low, p - 1)
            quicksort(p + 1, high)

    quicksort(0, len(S) - 1)
    return S, cont[0]

ya_ordenados = sorted(set(fila["valor"] for fila in base))   # valores distintos, ya ordenados
n = len(ya_ordenados)
_, c_primero = quicksort_primer_pivote(ya_ordenados)
_, c_medio = rapido(ya_ordenados)
print(f"n = {n}, entrada ya ordenada")
print(f"primer pivote: {c_primero:>6} comparaciones (n(n-1)/2 = {n * (n - 1) // 2})")
print(f"pivote medio:  {c_medio:>6} comparaciones (n log2 n = {n * log2(n):.0f})")

verificar("ambas versiones ordenan bien", (quicksort_primer_pivote(ya_ordenados)[0], rapido(ya_ordenados)[0]),
          (ya_ordenados, ya_ordenados))
verificar("primer pivote sobre datos ordenados es cuadrático", c_primero >= n * (n - 1) // 2, True)
verificar("el pivote medio evita el peor caso", c_medio < n * log2(n), True)

n = 371, entrada ya ordenada
primer pivote:  69005 comparaciones (n(n-1)/2 = 68635)
pivote medio:    2709 comparaciones (n log2 n = 3167)
Correcto: ambas versiones ordenan bien
Correcto: primer pivote sobre datos ordenados es cuadrático
Correcto: el pivote medio evita el peor caso


**Error común:** usar el primer elemento como pivote con datos que llegan ordenados, como las tablas oficiales, que suelen venir ordenadas por año o por código. Además de cuadrático, la recursión alcanza n niveles: con una lista de más de mil valores ya ordenados superaría el límite de recursión de Python (1,000 llamadas).

## 4. Comparación de los cinco algoritmos con 460 filas

Se ordenan las 460 filas de la base larga por `valor`. Como hay valores repetidos y selección y rápido no son estables, la clave es `(valor, posición original)`: así el orden correcto es único y cada resultado se puede comparar con `sorted()`. Se mide con tres entradas: el orden del archivo, ya ordenada e invertida. El tiempo es el mejor de tres ejecuciones y es orientativo.

In [7]:
filas = list(enumerate(base))                     # (posición original, fila)
clave = lambda par: (par[1]["valor"], par[0])
correcta = sorted(filas, key=clave)
entradas = {
    "archivo": filas,
    "ordenada": correcta,
    "invertida": list(reversed(correcta)),
}
n = len(filas)
print(f"n = {n} | n(n-1)/2 = {n * (n - 1) // 2:,} | n log2 n = {n * log2(n):,.0f}")
print(f"{'algoritmo':<11}" + "".join(f"{nombre:>24}" for nombre in entradas))

resultados = {}
for nombre, funcion in ALGORITMOS.items():
    celdas_fila = []
    for entrada, datos in entradas.items():
        ordenada, comparaciones, segundos = medir(funcion, datos, clave)
        resultados[(nombre, entrada)] = (ordenada == correcta, comparaciones)
        celdas_fila.append(f"{comparaciones:>10,} comp. {segundos * 1000:6.1f} ms")
    print(f"{nombre:<11}" + "".join(f"{celda:>24}" for celda in celdas_fila))

_, _, t_sorted = medir(lambda datos, k: (sorted(datos, key=k), 0), filas, clave)
print(f"{'sorted()':<11}{t_sorted * 1000:>21.2f} ms (Timsort, implementado en C)")

n = 460 | n(n-1)/2 = 105,570 | n log2 n = 4,069
algoritmo                   archivo                ordenada               invertida
burbuja       105,192 comp.   15.6 ms       459 comp.    0.1 ms   105,570 comp.   16.9 ms


selección     105,570 comp.   12.7 ms   105,570 comp.   12.9 ms   105,570 comp.   13.0 ms


inserción      71,515 comp.   10.1 ms       459 comp.    0.1 ms   105,570 comp.   15.7 ms
fusión          3,297 comp.    0.8 ms     1,976 comp.    0.5 ms     2,112 comp.    0.6 ms
rápido          6,160 comp.    0.8 ms     3,433 comp.    0.5 ms     4,028 comp.    0.6 ms
sorted()                    0.08 ms (Timsort, implementado en C)


In [8]:
for (nombre, entrada), (ok, _) in resultados.items():
    if entrada == "archivo":
        verificar(f"{nombre} coincide con sorted() en las {n} filas", ok, True)
verificar("los 15 ordenamientos coinciden con sorted()", all(ok for ok, _ in resultados.values()), True)
verificar("inserción hace n - 1 comparaciones con la lista ya ordenada", resultados[("inserción", "ordenada")][1], n - 1)
verificar("selección hace n(n-1)/2 comparaciones siempre",
          {resultados[("selección", e)][1] for e in entradas}, {n * (n - 1) // 2})
verificar("fusión y rápido comparan menos de 2 n log2 n veces en el archivo",
          all(resultados[(a, "archivo")][1] < 2 * n * log2(n) for a in ("fusión", "rápido")), True)

Correcto: burbuja coincide con sorted() en las 460 filas
Correcto: selección coincide con sorted() en las 460 filas
Correcto: inserción coincide con sorted() en las 460 filas
Correcto: fusión coincide con sorted() en las 460 filas
Correcto: rápido coincide con sorted() en las 460 filas
Correcto: los 15 ordenamientos coinciden con sorted()
Correcto: inserción hace n - 1 comparaciones con la lista ya ordenada
Correcto: selección hace n(n-1)/2 comparaciones siempre
Correcto: fusión y rápido comparan menos de 2 n log2 n veces en el archivo


| Algoritmo | Mejor caso | Caso promedio | Peor caso | Memoria adicional | Estable |
|---|---|---|---|---|---|
| Burbuja (con parada temprana) | O(n) | O(n²) | O(n²) | O(1) | sí |
| Selección | O(n²) | O(n²) | O(n²) | O(1) | no |
| Inserción | O(n) | O(n²) | O(n²) | O(1) | sí |
| Fusión | O(n log n) | O(n log n) | O(n log n) | O(n) | sí |
| Rápido | O(n log n) | O(n log n) | O(n²) | O(log n) de recursión | no |

**Clave:** con 460 filas, burbuja y selección rondan las 105,000 comparaciones en el orden del archivo (105,192 y 105,570) e inserción 71,515, mientras fusión y rápido se quedan entre 3,300 y 6,200: unas 17 a 32 veces menos. `sorted()` de Python es más rápido aún porque Timsort está escrito en C y aprovecha los tramos ya ordenados.

**Ampliación (no está en el PDF):** "estable" significa que dos elementos con la misma clave conservan su orden original. Por eso aquí se añadió la posición original a la clave: sin ella, selección y rápido podían devolver los repetidos en otro orden y la comparación con `sorted()` fallaría aunque el resultado estuviera ordenado.

## 5. Enfoque greedy: priorizar corregimientos con un presupuesto limitado

**Referencia:** [Unidad 30 — Enfoque greedy](../../../../01-python/semana-04/clase-2-greedy-divide-y-venceras/u30-enfoque-greedy.ipynb). Un algoritmo greedy toma en cada paso la opción que parece mejor en ese momento, comprueba que sea viable y no reconsidera. Así funcionan `coin_change` (la moneda más grande que cabe) y `activity_selection1` (la reunión que acaba antes).

Problema: con un presupuesto fijo, elegir qué corregimientos conectar para llevar internet al mayor número de viviendas que hoy no lo tienen. El beneficio es real (viviendas sin internet, Censo 2023). El criterio greedy es el beneficio por balboa: se ordenan los corregimientos por esa razón y se toman mientras quepan en el presupuesto.

**Ampliación (no está en el PDF):** los costos son un **supuesto ilustrativo, no un dato**. Se asume un costo de red de B/. 1,000 por km² de superficie más B/. 120 por vivienda conectada. Solo sirven para ilustrar el algoritmo; no provienen de ninguna fuente oficial ni de un presupuesto real.

In [9]:
COSTO_POR_KM2 = 1_000          # supuesto ilustrativo
COSTO_POR_VIVIENDA = 120       # supuesto ilustrativo

superficie = {c["id_corregimiento"]: c["superficie_km2"] for c in poblacion}
candidatos = []
for c in corregimientos:
    costo = round(COSTO_POR_KM2 * superficie[c["id"]] + COSTO_POR_VIVIENDA * c["sin_internet"])
    candidatos.append({"corregimiento": c["corregimiento"], "beneficio": c["sin_internet"], "costo": costo})

def priorizar_greedy(candidatos, presupuesto, criterio):
    """Ordena por el criterio (de mayor a menor) y toma cada candidato si todavía cabe."""
    orden, _ = fusion(candidatos, clave=lambda c: -criterio(c))
    elegidos, gasto = [], 0
    for c in orden:
        if gasto + c["costo"] <= presupuesto:     # viabilidad
            elegidos.append(c)
            gasto += c["costo"]
    return elegidos, gasto

por_razon = lambda c: c["beneficio"] / c["costo"]
elegidos, gasto = priorizar_greedy(candidatos, 1_000_000, por_razon)
print("Presupuesto ilustrativo: B/. 1,000,000")
for c in elegidos:
    print(f"  {c['corregimiento']:<28} {c['beneficio']:>5} viviendas  B/. {c['costo']:>9,}  "
          f"{1000 * por_razon(c):.2f} viviendas por cada B/. 1,000")
beneficio_greedy = sum(c["beneficio"] for c in elegidos)
print(f"Total: {beneficio_greedy:,} viviendas por B/. {gasto:,}")
verificar("el gasto no supera el presupuesto", gasto <= 1_000_000, True)

Presupuesto ilustrativo: B/. 1,000,000
  Penonomé (cabecera)           1323 viviendas  B/.   212,160  6.24 viviendas por cada B/. 1,000
  Pajonal                       1115 viviendas  B/.   197,500  5.65 viviendas por cada B/. 1,000
  Cañaveral                      828 viviendas  B/.   162,760  5.09 viviendas por cada B/. 1,000
  Chiguirí Arriba               1319 viviendas  B/.   262,280  5.03 viviendas por cada B/. 1,000
  Las Minas                      545 viviendas  B/.   136,400  4.00 viviendas por cada B/. 1,000
Total: 5,130 viviendas por B/. 971,100
Correcto: el gasto no supera el presupuesto


¿Es la mejor elección posible? Con 16 corregimientos hay 2¹⁶ = 65,536 subconjuntos, pocos para probarlos todos (fuerza bruta). Se compara el greedy por razón con el óptimo y con otro criterio greedy, el de mayor beneficio primero, en dos presupuestos.

In [10]:
def optimo_fuerza_bruta(candidatos, presupuesto):
    """Prueba los 2^n subconjuntos y devuelve el mayor beneficio que cabe en el presupuesto."""
    mejor = 0
    n = len(candidatos)
    for mascara in range(1 << n):
        costo = beneficio = 0
        for i in range(n):
            if mascara >> i & 1:
                costo += candidatos[i]["costo"]
                beneficio += candidatos[i]["beneficio"]
        if costo <= presupuesto and beneficio > mejor:
            mejor = beneficio
    return mejor

por_beneficio = lambda c: c["beneficio"]
comparacion = {}
print(f"{'presupuesto':>12} {'greedy razón':>13} {'greedy beneficio':>17} {'óptimo':>8}")
for presupuesto in (1_000_000, 500_000):
    g_razon = sum(c["beneficio"] for c in priorizar_greedy(candidatos, presupuesto, por_razon)[0])
    g_benef = sum(c["beneficio"] for c in priorizar_greedy(candidatos, presupuesto, por_beneficio)[0])
    optimo = optimo_fuerza_bruta(candidatos, presupuesto)
    comparacion[presupuesto] = (g_razon, g_benef, optimo)
    print(f"{presupuesto:>12,} {g_razon:>13,} {g_benef:>17,} {optimo:>8,}")

verificar("con B/. 1,000,000 el greedy por razón alcanza el óptimo", comparacion[1_000_000][0], comparacion[1_000_000][2])
verificar("con B/. 500,000 el greedy por razón no alcanza el óptimo", comparacion[500_000][0] < comparacion[500_000][2], True)
verificar("con B/. 1,000,000 la razón supera al mayor beneficio", comparacion[1_000_000][0] > comparacion[1_000_000][1], True)
verificar("con B/. 500,000 el mayor beneficio sí alcanza el óptimo", comparacion[500_000][1], comparacion[500_000][2])

 presupuesto  greedy razón  greedy beneficio   óptimo
   1,000,000         5,130             4,787    5,130


     500,000         2,438             2,642    2,642
Correcto: con B/. 1,000,000 el greedy por razón alcanza el óptimo
Correcto: con B/. 500,000 el greedy por razón no alcanza el óptimo
Correcto: con B/. 1,000,000 la razón supera al mayor beneficio
Correcto: con B/. 500,000 el mayor beneficio sí alcanza el óptimo


**Clave:** el greedy es rápido (ordenar y recorrer una vez: O(n log n)), pero no garantiza el óptimo. Este problema es una mochila 0/1: no se puede conectar medio corregimiento. Con B/. 500,000 el greedy por razón toma la cabecera y Pajonal (B/. 409,660) y los B/. 90,340 que sobran no alcanzan para ningún otro corregimiento; la cabecera con Chiguirí Arriba cuesta B/. 474,440 y conecta más viviendas. La fuerza bruta es exacta, pero cuesta O(2ⁿ): con 40 corregimientos serían más de un billón de subconjuntos.

**Error común:** suponer que un criterio greedy es siempre el mejor. La razón beneficio/costo gana con B/. 1,000,000 y el mayor beneficio gana con B/. 500,000: ninguno de los dos es óptimo en todos los casos. Para garantizar el óptimo hace falta explorar las combinaciones (fuerza bruta o programación dinámica, que se ve más adelante en el curso).

## 6. Divide y vencerás

**Referencia:** [Unidad 31 — Divide y vencerás](../../../../01-python/semana-04/clase-2-greedy-divide-y-venceras/u31-divide-y-venceras.ipynb). Dividir el problema en subproblemas del mismo tipo, resolverlos de forma recursiva (conquistar) y combinar sus resultados. La fusión y el ordenamiento rápido ya lo aplican; aquí se usa en tres problemas más.

### Suma y máximo-mínimo por mitades

`suma_dc` es la función de la unidad 31. `max_min_dc` busca a la vez el mayor y el menor dividiendo en mitades: en cada combinación hace dos comparaciones, y en un par de elementos, una. En total son unas 3n/2 - 2 comparaciones, frente a las hasta 2(n - 1) de `find_two` (unidad 24).

In [11]:
def suma_dc(lista):
    """Suma los elementos de una lista con divide y vencerás (unidad 31)."""
    if len(lista) == 0:
        return 0
    if len(lista) == 1:
        return lista[0]
    mitad = len(lista) // 2
    return suma_dc(lista[:mitad]) + suma_dc(lista[mitad:])

def max_min_dc(nums, low, high, cont):
    """Devuelve (mayor, menor) de nums[low..high]; cont[0] acumula las comparaciones."""
    if low == high:                               # caso base: un elemento
        return nums[low], nums[low]
    if high == low + 1:                           # caso base: dos elementos, una comparación
        cont[0] += 1
        return (nums[low], nums[high]) if nums[low] > nums[high] else (nums[high], nums[low])
    mid = (low + high) // 2                       # dividir
    mayor_izq, menor_izq = max_min_dc(nums, low, mid, cont)       # conquistar
    mayor_der, menor_der = max_min_dc(nums, mid + 1, high, cont)
    cont[0] += 2                                  # combinar
    return max(mayor_izq, mayor_der), min(menor_izq, menor_der)

habitantes = [c["pob_2023"] for c in poblacion]
cont = [0]
mayor, menor = max_min_dc(habitantes, 0, len(habitantes) - 1, cont)
_, _, comp_find_two = find_two(habitantes)
n = len(habitantes)
print("Suma por mitades:", suma_dc(habitantes))
print(f"Mayor: {mayor:,} | menor: {menor:,}")
print(f"Comparaciones: divide y vencerás {cont[0]} (3n/2 - 2 = {3 * n // 2 - 2}) | find_two {comp_find_two} (2(n-1) = {2 * (n - 1)})")

verificar("la suma por mitades da la población del distrito", suma_dc(habitantes), distrito["poblacion"])
verificar("mayor y menor coinciden con max() y min()", (mayor, menor), (max(habitantes), min(habitantes)))
verificar("divide y vencerás hace 3n/2 - 2 comparaciones con n = 16", cont[0], 3 * n // 2 - 2)
verificar("y compara menos que find_two en estos datos", cont[0] < comp_find_two, True)

Suma por mitades: 104326
Mayor: 25,836 | menor: 1,315
Comparaciones: divide y vencerás 22 (3n/2 - 2 = 22) | find_two 30 (2(n-1) = 30)
Correcto: la suma por mitades da la población del distrito
Correcto: mayor y menor coinciden con max() y min()
Correcto: divide y vencerás hace 3n/2 - 2 comparaciones con n = 16
Correcto: y compara menos que find_two en estos datos


### Subarreglo de suma máxima: el tramo de mayor actividad de las réplicas

Entre dos cortes consecutivos, el ritmo de réplicas es `(acumuladas finales - acumuladas iniciales) / días`. A cada ritmo se le resta el promedio de los seis ritmos, de modo que hay valores positivos (ritmo sobre el promedio) y negativos (bajo el promedio). La pregunta es cuál es el tramo de intervalos consecutivos con la mayor suma, es decir, el periodo seguido en que las réplicas fueron más intensas que lo habitual. Con divide y vencerás: el mejor tramo está entero en la mitad izquierda, entero en la derecha o cruza el centro; los dos primeros se resuelven de forma recursiva y el que cruza se calcula extendiéndose desde el centro hacia ambos lados.

**Ampliación (no está en el PDF):** el subarreglo de suma máxima es un problema clásico de divide y vencerás con costo O(n log n). Se comprueba contra la fuerza bruta, que prueba todos los tramos en O(n²). Los cortes son de prensa y el último (20 de noviembre) es aproximado; el promedio es simple, sin ponderar por los días de cada intervalo.

In [12]:
MESES = ("ene", "feb", "mar", "abr", "may", "jun", "jul", "ago", "sep", "oct", "nov", "dic")
etiqueta = lambda f: f"{f.day} {MESES[f.month - 1]}"

cortes = [(date.fromisoformat(fila["fecha"]), fila["replicas_acumuladas"]) for fila in replicas]
intervalos, ritmos = [], []
for (f0, r0), (f1, r1) in zip(cortes, cortes[1:]):
    intervalos.append(f"{etiqueta(f0)} a {etiqueta(f1)}")
    ritmos.append((r1 - r0) / (f1 - f0).days)
promedio = sum(ritmos) / len(ritmos)
diferencias = [round(r - promedio, 1) for r in ritmos]

def tramo_cruzado(a, low, mid, high):
    """Mejor tramo que contiene a[mid] y a[mid + 1]: (suma, inicio, fin)."""
    suma, mejor_izq, inicio = 0, None, mid
    for i in range(mid, low - 1, -1):
        suma += a[i]
        if mejor_izq is None or suma > mejor_izq:
            mejor_izq, inicio = suma, i
    suma, mejor_der, fin = 0, None, mid + 1
    for j in range(mid + 1, high + 1):
        suma += a[j]
        if mejor_der is None or suma > mejor_der:
            mejor_der, fin = suma, j
    return mejor_izq + mejor_der, inicio, fin

def subarreglo_maximo(a, low, high):
    """Divide y vencerás: (suma máxima, inicio, fin) de a[low..high]."""
    if low == high:                                   # caso base: un intervalo
        return a[low], low, high
    mid = (low + high) // 2                           # dividir
    izquierdo = subarreglo_maximo(a, low, mid)        # conquistar
    derecho = subarreglo_maximo(a, mid + 1, high)
    cruzado = tramo_cruzado(a, low, mid, high)
    return max(izquierdo, derecho, cruzado, key=lambda t: t[0])   # combinar

def subarreglo_fuerza_bruta(a):
    mejor = (a[0], 0, 0)
    for i in range(len(a)):
        suma = 0
        for j in range(i, len(a)):
            suma += a[j]
            if suma > mejor[0]:
                mejor = (suma, i, j)
    return mejor

print(f"Ritmo promedio de los seis intervalos: {promedio:.1f} réplicas por día")
for intervalo, r, d in zip(intervalos, ritmos, diferencias):
    print(f"{intervalo:<12} {r:6.1f} réplicas por día  ({d:+6.1f} frente al promedio)")
suma, inicio, fin = subarreglo_maximo(diferencias, 0, len(diferencias) - 1)
print(f"Mejor tramo: del {intervalos[inicio].split(' a ')[0]} al {intervalos[fin].split(' a ')[1]}, suma {suma:+.1f}")

verificar("divide y vencerás coincide con la fuerza bruta",
          (round(suma, 6), inicio, fin), tuple(round(x, 6) if k == 0 else x for k, x in enumerate(subarreglo_fuerza_bruta(diferencias))))
verificar("el tramo cubre los tres primeros intervalos", (inicio, fin), (0, 2))
verificar("la suma del tramo es la suma de sus diferencias", round(suma, 6), round(sum(diferencias[inicio:fin + 1]), 6))

Ritmo promedio de los seis intervalos: 36.9 réplicas por día
10 oct a 16 oct  101.8 réplicas por día  ( +65.0 frente al promedio)
16 oct a 2 nov   30.9 réplicas por día  (  -6.0 frente al promedio)
2 nov a 6 nov   45.2 réplicas por día  (  +8.4 frente al promedio)
6 nov a 9 nov   10.3 réplicas por día  ( -26.5 frente al promedio)
9 nov a 17 nov   21.6 réplicas por día  ( -15.3 frente al promedio)
17 nov a 20 nov   11.3 réplicas por día  ( -25.5 frente al promedio)
Mejor tramo: del 10 oct al 6 nov, suma +67.4
Correcto: divide y vencerás coincide con la fuerza bruta
Correcto: el tramo cubre los tres primeros intervalos
Correcto: la suma del tramo es la suma de sus diferencias


## Conclusiones

- Los cinco algoritmos de ordenamiento del curso producen exactamente el mismo orden que `sorted()` sobre los 16 corregimientos y sobre las 460 filas de la base larga, en el orden del archivo, ya ordenadas e invertidas.
- Burbuja y selección hacen unas 105,000 comparaciones con 460 filas e inserción unas 71,500; fusión y rápido, entre 3,300 y 6,200. Inserción y burbuja con parada temprana son lineales si la entrada ya está ordenada; selección nunca lo es.
- El pivote importa: con datos ya ordenados, el primer pivote vuelve cuadrático el ordenamiento rápido; el pivote central lo evita.
- En Tulú, Río Indio, Boca de Tucué y Riecito, más del 98 % de las viviendas no tiene internet; en la cabecera, el 17.5 %. Entre 2010 y 2023, El Coco fue la unidad que más creció.
- El greedy por razón beneficio/costo alcanza el óptimo con el presupuesto ilustrativo de B/. 1,000,000, pero no con B/. 500,000, donde gana el criterio de mayor beneficio: en una mochila 0/1 ningún criterio greedy garantiza el óptimo.
- Divide y vencerás suma la población del distrito (104,326), encuentra el mayor y el menor con 22 comparaciones y ubica el tramo de mayor actividad de las réplicas: del 10 de octubre al 6 de noviembre de 2026, con ritmos sobre el promedio de 36.9 réplicas por día.

[Proyecto](../../README.md) | [Anterior](s03-pilas-colas-busqueda-hash.ipynb) | [Siguiente](s05-dinamica-backtracking-modulos-pandas.ipynb)